##### ARTI 560 - Computer Vision

## Object Detection Using YOLOV8 with KerasCV - Exercise 

### Objective

In this exercise, you wil learn how to use the pre-trained YOLOV8 model from KerasCV to detect objects in images.

You will:

1. Load the pre-trained YOLOV8 model from KerasCV using the [Pascal VOC preset](https://www.kaggle.com/models/keras/yolov8)

2. Load 5 images for different classes in [Pascal VOC 2012 dataset](https://datasetninja.com/pascal-voc-2012) and convert it into a NumPy array suitable for the model.

3. Resize the images before inference to match the model’s expected input size using:
    ```
    inference_resizing = keras_cv.layers.Resizing(
        640, 640, pad_to_aspect_ratio=True, bounding_box_format="xywh"
    )
    ```

    **Note:** Resizing ensures that the images fit the model input, maintains aspect ratio, and correctly adjusts bounding boxes.

4. Run the YOLOV8 detector on each image to predict bounding boxes, class labels, and confidence scores.

5. Visualize the predictions by drawing the bounding boxes and labels on the images.

6. Record for each image:

    - Which objects were detected correctly

    - The confidence scores of the detections

    - Any missed or incorrectly labeled objects

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import tensorflow as tf
import tensorflow_datasets as tfds
import keras
import keras_cv
from IPython.display import display

# Pascal VOC class mapping used by the pretrained preset
class_ids = [
    "Aeroplane", "Bicycle", "Bird", "Boat", "Bottle",
    "Bus", "Car", "Cat", "Chair", "Cow",
    "Dining Table", "Dog", "Horse", "Motorbike", "Person",
    "Potted Plant", "Sheep", "Sofa", "Train", "Tvmonitor",
]
class_mapping = dict(zip(range(len(class_ids)), class_ids))

# Load the pretrained YOLOV8 Pascal VOC model
prediction_decoder = keras_cv.layers.NonMaxSuppression(
    bounding_box_format="xywh",
    from_logits=True,
    iou_threshold=0.5,
    confidence_threshold=0.5,
)

model = keras_cv.models.YOLOV8Detector.from_preset(
    "yolo_v8_m_pascalvoc",
    bounding_box_format="xywh",
    prediction_decoder=prediction_decoder,
)

# Resize images for inference exactly as requested in the exercise
inference_resizing = keras_cv.layers.Resizing(
    640, 640, pad_to_aspect_ratio=True, bounding_box_format="xywh"
)

# Load Pascal VOC 2012 examples and pick 5 images with different primary classes.
# To make evaluation easier, we prefer images whose annotations contain only one class.
train_ds = tfds.load("voc/2012", split="train", shuffle_files=False)

selected_examples = []
used_primary_labels = set()

for sample in tfds.as_numpy(train_ds):
    labels = sample["objects"]["label"]
    unique_labels = sorted(set(labels.tolist()))

    if len(unique_labels) != 1:
        continue

    primary_label = int(unique_labels[0])
    if primary_label in used_primary_labels:
        continue

    image = sample["image"]
    boxes_rel_yxyx = sample["objects"]["bbox"]

    # Skip tiny images just in case
    if image.shape[0] < 100 or image.shape[1] < 100:
        continue

    selected_examples.append(
        {
            "image": image,
            "gt_label_ids": unique_labels,
            "gt_labels": [class_mapping[i] for i in unique_labels],
            "boxes_rel_yxyx": boxes_rel_yxyx,
        }
    )
    used_primary_labels.add(primary_label)

    if len(selected_examples) == 5:
        break

if len(selected_examples) < 5:
    raise RuntimeError("Could not find 5 suitable Pascal VOC 2012 examples.")

# Convert images to a NumPy batch suitable for the model
raw_images = [example["image"] for example in selected_examples]
resized_images = []
for image in raw_images:
    resized = inference_resizing(np.expand_dims(image, axis=0))
    resized_images.append(np.array(resized[0]).astype("uint8"))

image_batch = np.stack(resized_images, axis=0)

# Run inference
predictions = model.predict(image_batch, verbose=0)

# Robustly get score tensor name across versions
score_key = None
for key in ["confidence", "confidence_scores", "scores"]:
    if isinstance(predictions, dict) and key in predictions:
        score_key = key
        break

pred_boxes = np.array(predictions["boxes"])
pred_classes = np.array(predictions["classes"])
pred_scores = np.array(predictions[score_key]) if score_key is not None else np.ones_like(pred_classes, dtype=float)

# Helper to draw predicted boxes

def plot_predictions(image, boxes, classes, scores, title, threshold=0.5):
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(image.astype("uint8"))
    ax.set_title(title)
    ax.axis("off")

    for box, cls_id, score in zip(boxes, classes, scores):
        if score < threshold or cls_id < 0:
            continue

        x, y, w, h = box
        rect = patches.Rectangle(
            (x, y), w, h, linewidth=2, edgecolor="lime", facecolor="none"
        )
        ax.add_patch(rect)
        class_name = class_mapping.get(int(cls_id), f"Class {int(cls_id)}")
        ax.text(
            x,
            max(y - 5, 5),
            f"{class_name}: {score:.2f}",
            fontsize=10,
            bbox=dict(facecolor="black", alpha=0.7),
            color="white",
        )

    plt.show()

# Visualize predictions and create results table
results = []
threshold = 0.5

for i, example in enumerate(selected_examples):
    gt_labels = example["gt_labels"]

    valid_mask = (pred_scores[i] >= threshold) & (pred_classes[i] >= 0)
    valid_boxes = pred_boxes[i][valid_mask]
    valid_classes = pred_classes[i][valid_mask].astype(int)
    valid_scores = pred_scores[i][valid_mask]

    detected_labels = [class_mapping[int(c)] for c in valid_classes]
    detected_with_scores = [
        f"{class_mapping[int(c)]} ({float(s):.2f})"
        for c, s in zip(valid_classes, valid_scores)
    ]

    correct_detected = sorted(set(gt_labels).intersection(set(detected_labels)))
    missed_objects = sorted(set(gt_labels) - set(detected_labels))
    incorrect_labels = sorted(set(detected_labels) - set(gt_labels))

    plot_predictions(
        image_batch[i],
        valid_boxes,
        valid_classes,
        valid_scores,
        title=f"Image {i + 1} | Ground Truth: {', '.join(gt_labels)}",
        threshold=threshold,
    )

    results.append(
        {
            "Image": f"Image {i + 1}",
            "Ground Truth Class(es)": ", ".join(gt_labels),
            "Detected Correctly": ", ".join(correct_detected) if correct_detected else "None",
            "Confidence Scores": ", ".join(detected_with_scores) if detected_with_scores else "No detections",
            "Missed Objects": ", ".join(missed_objects) if missed_objects else "None",
            "Incorrect Labels": ", ".join(incorrect_labels) if incorrect_labels else "None",
        }
    )

results_df = pd.DataFrame(results)
display(results_df)



### Record Your Results

Run the code cell above to load 5 Pascal VOC 2012 images, perform YOLOV8 inference, visualize the predicted bounding boxes, and automatically generate the results table.
